# Polaris Privilege Matrix — Diagnostic Suite

Determines the **minimum fine-grained privilege** required for each namespace/table/view
mutation under Apache Polaris 1.3.0, producing a verified authorization truth table.

Implements `polaris-privilege-matrix-plan.md`. Built on the Phase-0 scaff/worker lifecycle
(`case_scaffold`, `worker_principal`). Every challenge runs as a single-use **worker**
principal holding exactly one candidate privilege; root only bootstraps/destroys.

**Six-way outcome taxonomy** (the safety net the source strategy lacked):
```
AUTHORIZED      2xx (or lag-500 commit / 409 already-exists) → privilege sufficient
BLOCKED_PRIV    403 "is not authorized"      → privilege insufficient (clean gate)
BLOCKED_OPA     403 OPA/ext_authz            → token/role not activated  (HARNESS BUG)
BLOCKED_CONFIG  403 "Unable to purge"        → config gate leaked into a drop (HARNESS BUG)
GRANT_INVALID   grant PUT 400/invalid        → privilege name not accepted by this build
LAG_500         500 metadata/NPE (no commit) → read-after-write lag (retry inconclusive)
```


In [2]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────────
import sys, pathlib
_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import sys, time, uuid
import importlib, polaris_test_utils
importlib.reload(polaris_test_utils)
from polaris_test_utils import *
require_not_prod("privilege-matrix mutations")  # PROD-safe: refuses to run against company PROD

tok = root_token()
_RAW_SETTLE = 0.6                      # settle after a read-after-write lag-500
NS = lambda cat: f"ns-{cat}"           # per-catalog namespace (Vector B: no flat strings)
print("✅ privilege-matrix harness loaded")

🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
✅ privilege-matrix harness loaded


In [3]:
# ── MATRIX SPEC ──────────────────────────────────────────────────────────────
# Each spec is one (target, action). build(cat, hdr, ctx) scaffolds the prerequisite
# AS THE SCAFF principal; challenge(cat, hdr, ctx) fires the action AS THE WORKER.
# `candidates` are tested ONE AT A TIME; `coarse` masters are also tested to verify
# the cascade. ✓ = privilege name already confirmed accepted by this build.

def _tbl_body(cat, ns, tbl):
    return {"name": tbl, "location": f"s3a://{BUCKET}/{cat}/{ns}/{tbl}/",
            "schema": {"type":"struct","fields":[
                {"id":1,"name":"id","type":"long","required":True},
                {"id":2,"name":"value","type":"string","required":False}]}}

def _view_body(cat, ns, vw):
    return {"name": vw, "default-namespace":[ns],
            "schema":{"type":"struct","schema-id":0,
                      "fields":[{"id":1,"name":"id","type":"long","required":True}]},
            "view-version":{"version-id":1,"timestamp-ms":int(time.time()*1000),
                            "schema-id":0,"default-namespace":[ns],"summary":{"engine":"spark"},
                            "representations":[{"type":"sql","sql":"SELECT 1 AS id","dialect":"spark"}]},
            "location": f"s3a://{BUCKET}/{cat}/{ns}/{vw}/"}

# build helpers (run as scaff) ------------------------------------------------
def _mk_ns(cat, hdr):
    requests.post(f"{BASE_CAT}/{cat}/namespaces", headers=hdr,
                  json={"namespace":[NS(cat)], "properties":{}})
def _mk_tbl(cat, hdr, ctx):
    r = requests.post(f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables", headers=hdr,
                      json=_tbl_body(cat, NS(cat), ctx["entity"]))
    try: ctx["uuid"] = r.json().get("metadata",{}).get("table-uuid")
    except Exception: ctx["uuid"] = None
def _mk_view(cat, hdr, ctx):
    requests.post(f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/views", headers=hdr,
                  json=_view_body(cat, NS(cat), ctx["entity"]))

# specs -----------------------------------------------------------------------
SPECS = [
  # NAMESPACE
  dict(key="ns_create", target="Namespace", action="CREATE", drop=False,
       candidates=["NAMESPACE_CREATE"], coarse=["CATALOG_MANAGE_CONTENT"],
       build=lambda cat,hdr,ctx: None,
       challenge=lambda cat,hdr,ctx: requests.post(
           f"{BASE_CAT}/{cat}/namespaces", headers=hdr,
           json={"namespace":[NS(cat)], "properties":{}})),
  dict(key="ns_list", target="Namespace", action="LIST", drop=False,
       candidates=["NAMESPACE_LIST"], coarse=["CATALOG_MANAGE_METADATA","CATALOG_MANAGE_CONTENT"],
       build=lambda cat,hdr,ctx: _mk_ns(cat,hdr),
       challenge=lambda cat,hdr,ctx: requests.get(
           f"{BASE_CAT}/{cat}/namespaces", headers=hdr)),
  dict(key="ns_drop", target="Namespace", action="DROP", drop=True,
       candidates=["NAMESPACE_DROP"], coarse=["CATALOG_MANAGE_CONTENT"],
       build=lambda cat,hdr,ctx: _mk_ns(cat,hdr),
       challenge=lambda cat,hdr,ctx: requests.delete(
           f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}", headers=hdr)),
  # TABLE
  dict(key="tbl_create", target="Table", action="CREATE", drop=False,
       candidates=["TABLE_CREATE"], coarse=["CATALOG_MANAGE_CONTENT"],
       build=lambda cat,hdr,ctx: _mk_ns(cat,hdr),
       challenge=lambda cat,hdr,ctx: requests.post(
           f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables", headers=hdr,
           json=_tbl_body(cat, NS(cat), ctx["entity"]))),
  dict(key="tbl_list", target="Table", action="LIST", drop=False,
       candidates=["TABLE_LIST"], coarse=["CATALOG_MANAGE_METADATA","CATALOG_MANAGE_CONTENT"],
       build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_tbl(cat,hdr,ctx)),
       challenge=lambda cat,hdr,ctx: requests.get(
           f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables", headers=hdr)),
  dict(key="tbl_read", target="Table", action="READ", drop=False,
       candidates=["TABLE_READ_DATA","TABLE_READ_PROPERTIES"],
       coarse=["CATALOG_MANAGE_METADATA","CATALOG_MANAGE_CONTENT"],
       build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_tbl(cat,hdr,ctx)),
       challenge=lambda cat,hdr,ctx: requests.get(
           f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables/{ctx['entity']}", headers=hdr)),
  dict(key="tbl_commit", target="Table", action="COMMIT", drop=False,
       candidates=["TABLE_WRITE_DATA","TABLE_WRITE_PROPERTIES"],
       coarse=["CATALOG_MANAGE_METADATA","CATALOG_MANAGE_CONTENT"],
       build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_tbl(cat,hdr,ctx)),
       challenge=lambda cat,hdr,ctx: requests.post(
           f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables/{ctx['entity']}", headers=hdr,
           json={"requirements":[], "updates":[
               {"action":"set-properties","updates":{"probe":"1"}}]})),
  dict(key="tbl_drop", target="Table", action="DROP", drop=True,
       candidates=["TABLE_DROP"], coarse=["CATALOG_MANAGE_CONTENT"],
       build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_tbl(cat,hdr,ctx)),
       challenge=lambda cat,hdr,ctx: requests.delete(
           f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables/{ctx['entity']}", headers=hdr)),
  # VIEW
  dict(key="view_create", target="View", action="CREATE", drop=False,
       candidates=["VIEW_CREATE"], coarse=["CATALOG_MANAGE_CONTENT"],
       build=lambda cat,hdr,ctx: _mk_ns(cat,hdr),
       challenge=lambda cat,hdr,ctx: requests.post(
           f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/views", headers=hdr,
           json=_view_body(cat, NS(cat), ctx["entity"]))),
  dict(key="view_get", target="View", action="GET", drop=False,
       candidates=["VIEW_LIST","VIEW_READ_PROPERTIES"],
       coarse=["CATALOG_MANAGE_METADATA","CATALOG_MANAGE_CONTENT"],
       build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_view(cat,hdr,ctx)),
       challenge=lambda cat,hdr,ctx: requests.get(
           f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/views/{ctx['entity']}", headers=hdr)),
  dict(key="view_drop", target="View", action="DROP", drop=True,
       candidates=["VIEW_DROP"], coarse=["CATALOG_MANAGE_METADATA","CATALOG_MANAGE_CONTENT"],
       build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_view(cat,hdr,ctx)),
       challenge=lambda cat,hdr,ctx: requests.delete(
           f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/views/{ctx['entity']}", headers=hdr)),
]
print(f"✅ {len(SPECS)} specs; "
      f"{sum(len(s['candidates'])+len(s['coarse']) for s in SPECS)} (spec×privilege) probes")

✅ 11 specs; 31 (spec×privilege) probes


In [4]:
# ── classifier ───────────────────────────────────────────────────────────────
def _is_lag(r):
    b = r.text.lower()
    return r.status_code == 500 and ("metadata" in b or "nullpointer" in b)

def classify(r):
    sc, b = r.status_code, r.text.lower()
    if sc in (200, 201, 204):                       return "AUTHORIZED"
    if sc == 409 or "already exists" in b:           return "AUTHORIZED"   # create lag-committed
    if _is_lag(r):                                   return "LAG_500"
    if sc == 403:
        if "unable to purge" in b:                   return "BLOCKED_CONFIG"   # harness bug
        if "is not authorized" in b or "not authorized for op" in b: return "BLOCKED_PRIV"
        if "opa" in b or "ext_authz" in b or "denied by policy" in b: return "BLOCKED_OPA"  # harness bug
        return "BLOCKED_PRIV"                         # unlabeled 403 → treat as privilege
    if sc == 404:                                    return "NOT_FOUND_404"   # setup issue
    return f"HTTP_{sc}"

# ── worker provisioning WITH grant-response capture (for GRANT_INVALID) ───────
def provision_worker(cat, privilege):
    """Create a single-use worker holding exactly `privilege`. Returns
    (worker_header_or_None, grant_outcome) where grant_outcome is 'ok' or
    'GRANT_INVALID'. Token is obtained AFTER the grant so it reflects it."""
    P, PR, CR = f"{cat}-worker-prin", f"{cat}-worker-prole", f"{cat}-worker-crole"
    requests.delete(f"{BASE_MGMT}/principals/{P}", headers=h(tok))
    requests.delete(f"{BASE_MGMT}/principal-roles/{PR}", headers=h(tok))
    rp   = create_principal(P, token=tok).json()
    cid  = rp.get("principal", {}).get("clientId")     or rp.get("clientId")
    csec = rp.get("credentials", {}).get("clientSecret") or rp.get("clientSecret")
    create_principal_role(PR, token=tok)
    create_catalog_role(catalog=cat, name=CR, token=tok)
    assign_catalog_role_to_principal_role(catalog=cat, pr=PR, cr=CR, token=tok)
    assign_principal_role_to_principal(principal=P, pr=PR, token=tok)
    # grant the single candidate privilege, capture the response
    gr = grant_privilege(catalog=cat, cr=CR, privilege=privilege, token=tok)
    gb = gr.text.lower()
    granted = (gr.status_code < 400
               or "already exists" in gb or "23505" in gb or "duplicate key" in gb)
    if not granted:
        # privilege name rejected by this build (400 / invalid enum / unknown)
        return None, "GRANT_INVALID"
    # token AFTER grant, scoped to the worker's principal-role
    tr = get_token(client_id=cid, client_secret=csec, scope=f"PRINCIPAL_ROLE:{PR}")
    if tr.status_code != 200:
        return None, "ok"   # token issue surfaces as BLOCKED_OPA downstream
    wtok = tr.json().get("access_token")
    return {"Authorization": f"Bearer {wtok}", "Polaris-Realm": REALM,
            "Content-Type": "application/json"}, "ok"

# ── single probe: one (spec, privilege) ──────────────────────────────────────
def probe_privilege(spec, privilege, verbose=False):
    cat = f"privmx-{spec['key'].replace('_','')}-{privilege.lower().replace('_','')}-{uuid.uuid4().hex[:4]}"
    # fresh catalog (root). For DROP actions, NEUTRALIZE the config gate so the
    # only thing that can 403 the drop is privilege (gap 1).
    props = {}
    if spec["drop"]:
        props = {"polaris.config.drop-with-purge.enabled": "true",
                 "polaris.config.purge-view-metadata-on-drop": "false"}
    create_catalog(cat, token=tok, properties=props)
    ctx = {"entity": f"e{uuid.uuid4().hex[:6]}"}
    outcome, detail = "ERROR", ""
    try:
        with case_scaffold(cat, build=lambda hdr: spec["build"](cat, hdr, ctx),
                           teardown=None, root=tok, destroy_scaff=True):
            whdr, gstat = provision_worker(cat, privilege)
            if gstat == "GRANT_INVALID":
                outcome, detail = "GRANT_INVALID", f"grant {privilege} rejected"
            elif whdr is None:
                outcome, detail = "BLOCKED_OPA", "worker token not issued"
            else:
                r = spec["challenge"](cat, whdr, ctx)
                if _is_lag(r):                       # one retry/settle on lag-500
                    time.sleep(_RAW_SETTLE)
                    r = spec["challenge"](cat, whdr, ctx)
                outcome, detail = classify(r), f"[{r.status_code}] {r.text[:90]}"
    except Exception as e:
        outcome, detail = "ERROR", str(e)[:120]
    if verbose:
        print(f"  {spec['key']:12} {privilege:28} → {outcome:14} {detail}")
    return outcome

# ── VALIDATION GATE: reproduce the known C9 result before trusting the matrix ──
print("Validation gate (must pass before running the full matrix):")
vs = next(s for s in SPECS if s["key"] == "view_drop")
v_auth    = probe_privilege(vs, "VIEW_DROP",  verbose=True)   # expect AUTHORIZED
v_blocked = probe_privilege(vs, "VIEW_LIST",  verbose=True)   # expect BLOCKED_PRIV (≈ C9)
ok = (v_auth == "AUTHORIZED" and v_blocked == "BLOCKED_PRIV")
print(f"\n{'✅ VALIDATION PASSED' if ok else '❌ VALIDATION FAILED'}: "
      f"VIEW_DROP→{v_auth}, VIEW_LIST→{v_blocked}")
if not ok:
    print("   Do NOT trust the matrix yet — check token scope / config neutralization.")
    print("   A BLOCKED_OPA or BLOCKED_CONFIG here means a harness bug, not a privilege result.")

Validation gate (must pass before running the full matrix):
  view_drop    VIEW_DROP                    → AUTHORIZED     [204] 
  view_drop    VIEW_LIST                    → BLOCKED_PRIV   [403] {"error":{"message":"Principal 'privmx-viewdrop-viewlist-d121-worker-prin' with activated 

✅ VALIDATION PASSED: VIEW_DROP→AUTHORIZED, VIEW_LIST→BLOCKED_PRIV


In [5]:
# ── run the full matrix ──────────────────────────────────────────────────────
MATRIX = {}        # (spec_key, action_label, privilege) -> outcome
print("Running privilege matrix...\n")
for spec in SPECS:
    label = f"{spec['target']}.{spec['action']}"
    for priv in spec["candidates"] + spec["coarse"]:
        outcome = probe_privilege(spec, priv)
        MATRIX[(spec["key"], label, priv)] = outcome
        tag = {"AUTHORIZED":"✅","BLOCKED_PRIV":"🔒","GRANT_INVALID":"∅",
               "BLOCKED_OPA":"⚠️OPA","BLOCKED_CONFIG":"⚠️CFG","LAG_500":"…",
               "NOT_FOUND_404":"404"}.get(outcome, outcome)
        print(f"  {label:16} {priv:28} → {tag}  ({outcome})")
print(f"\n✅ matrix complete: {len(MATRIX)} probes")

Running privilege matrix...

  Namespace.CREATE NAMESPACE_CREATE             → ✅  (AUTHORIZED)
  Namespace.CREATE CATALOG_MANAGE_CONTENT       → ✅  (AUTHORIZED)
  Namespace.LIST   NAMESPACE_LIST               → ✅  (AUTHORIZED)
  Namespace.LIST   CATALOG_MANAGE_METADATA      → ✅  (AUTHORIZED)
  Namespace.LIST   CATALOG_MANAGE_CONTENT       → ✅  (AUTHORIZED)
  Namespace.DROP   NAMESPACE_DROP               → ✅  (AUTHORIZED)
  Namespace.DROP   CATALOG_MANAGE_CONTENT       → ✅  (AUTHORIZED)
  Table.CREATE     TABLE_CREATE                 → ✅  (AUTHORIZED)
  Table.CREATE     CATALOG_MANAGE_CONTENT       → ✅  (AUTHORIZED)
  Table.LIST       TABLE_LIST                   → ✅  (AUTHORIZED)
  Table.LIST       CATALOG_MANAGE_METADATA      → ✅  (AUTHORIZED)
  Table.LIST       CATALOG_MANAGE_CONTENT       → ✅  (AUTHORIZED)
  Table.READ       TABLE_READ_DATA              → ✅  (AUTHORIZED)
  Table.READ       TABLE_READ_PROPERTIES        → ✅  (AUTHORIZED)
  Table.READ       CATALOG_MANAGE_METADATA     

In [6]:
# ── render the truth table (Markdown) ────────────────────────────────────────
from IPython.display import Markdown, display

# columns = union of all privileges tested, ordered: fine-grained then coarse masters
fine  = []
for s in SPECS:
    for p in s["candidates"]:
        if p not in fine: fine.append(p)
coarse = ["CATALOG_MANAGE_METADATA", "CATALOG_MANAGE_CONTENT"]
cols = fine + [c for c in coarse if c not in fine]

SYM = {"AUTHORIZED":"✅","BLOCKED_PRIV":"🔒","GRANT_INVALID":"∅",
       "BLOCKED_OPA":"⚠️OPA","BLOCKED_CONFIG":"⚠️CFG","LAG_500":"…","NOT_FOUND_404":"404"}

rows = []
seen = []
for s in SPECS:
    label = f"{s['target']}.{s['action']}"
    if label in seen: continue
    seen.append(label)
    cells_md = []
    for c in cols:
        out = MATRIX.get((s["key"], label, c))
        cells_md.append(SYM.get(out, "") if out else "·")
    rows.append((label, cells_md))

hdr = "| Action | " + " | ".join(cols) + " |"
sep = "|" + "---|"*(len(cols)+1)
body = "\n".join("| " + lbl + " | " + " | ".join(cs) + " |" for lbl,cs in rows)
legend = ("\n**Legend:** ✅ AUTHORIZED · 🔒 BLOCKED_PRIV · ∅ GRANT_INVALID · "
          "… LAG_500 · ⚠️OPA/⚠️CFG = harness bug · · not tested")
display(Markdown(f"### Polaris 1.3.0 — Privilege Truth Table\n\n{hdr}\n{sep}\n{body}\n{legend}"))

### Polaris 1.3.0 — Privilege Truth Table

| Action | NAMESPACE_CREATE | NAMESPACE_LIST | NAMESPACE_DROP | TABLE_CREATE | TABLE_LIST | TABLE_READ_DATA | TABLE_READ_PROPERTIES | TABLE_WRITE_DATA | TABLE_WRITE_PROPERTIES | TABLE_DROP | VIEW_CREATE | VIEW_LIST | VIEW_READ_PROPERTIES | VIEW_DROP | CATALOG_MANAGE_METADATA | CATALOG_MANAGE_CONTENT |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Namespace.CREATE | ✅ | · | · | · | · | · | · | · | · | · | · | · | · | · | · | ✅ |
| Namespace.LIST | · | ✅ | · | · | · | · | · | · | · | · | · | · | · | · | ✅ | ✅ |
| Namespace.DROP | · | · | ✅ | · | · | · | · | · | · | · | · | · | · | · | · | ✅ |
| Table.CREATE | · | · | · | ✅ | · | · | · | · | · | · | · | · | · | · | · | ✅ |
| Table.LIST | · | · | · | · | ✅ | · | · | · | · | · | · | · | · | · | ✅ | ✅ |
| Table.READ | · | · | · | · | · | ✅ | ✅ | · | · | · | · | · | · | · | ✅ | ✅ |
| Table.COMMIT | · | · | · | · | · | · | · | ✅ | ✅ | · | · | · | · | · | ✅ | ✅ |
| Table.DROP | · | · | · | · | · | · | · | · | · | ✅ | · | · | · | · | · | ✅ |
| View.CREATE | · | · | · | · | · | · | · | · | · | · | ✅ | · | · | · | · | ✅ |
| View.GET | · | · | · | · | · | · | · | · | · | · | · | 🔒 | ✅ | · | ✅ | ✅ |
| View.DROP | · | · | · | · | · | · | · | · | · | · | · | · | · | ✅ | ✅ | ✅ |

**Legend:** ✅ AUTHORIZED · 🔒 BLOCKED_PRIV · ∅ GRANT_INVALID · … LAG_500 · ⚠️OPA/⚠️CFG = harness bug · · not tested

In [7]:
# ── summary: minimal privilege per action, discovered vocabulary, bug flags ───
print("MINIMAL PRIVILEGE PER ACTION (first privilege that AUTHORIZED):\n")
seen=set()
for s in SPECS:
    label=f"{s['target']}.{s['action']}"
    if label in seen: continue
    seen.add(label)
    authorizing=[p for p in (s["candidates"]+s["coarse"])
                 if MATRIX.get((s["key"],label,p))=="AUTHORIZED"]
    fine_auth=[p for p in s["candidates"]
               if MATRIX.get((s["key"],label,p))=="AUTHORIZED"]
    minimal = fine_auth[0] if fine_auth else (authorizing[0] if authorizing else "— none authorized —")
    print(f"  {label:16} minimal: {minimal:26} all authorizing: {authorizing}")

print("\nDISCOVERED PRIVILEGE VOCABULARY (this build):")
valid   = sorted({p for (k,l,p),o in MATRIX.items() if o!="GRANT_INVALID"})
invalid = sorted({p for (k,l,p),o in MATRIX.items() if o=="GRANT_INVALID"})
print(f"  ✅ accepted at grant time : {valid}")
print(f"  ∅ rejected (GRANT_INVALID): {invalid}")

flags=[(k,l,p,o) for (k,l,p),o in MATRIX.items() if o in ("BLOCKED_OPA","BLOCKED_CONFIG","LAG_500","ERROR")]
print("\nHARNESS-BUG / INCONCLUSIVE FLAGS (should be empty in a clean run):")
if flags:
    for k,l,p,o in flags: print(f"  ⚠️ {l} / {p} → {o}")
else:
    print("  ✅ none — every cell is a clean AUTHORIZED / BLOCKED_PRIV / GRANT_INVALID")

MINIMAL PRIVILEGE PER ACTION (first privilege that AUTHORIZED):

  Namespace.CREATE minimal: NAMESPACE_CREATE           all authorizing: ['NAMESPACE_CREATE', 'CATALOG_MANAGE_CONTENT']
  Namespace.LIST   minimal: NAMESPACE_LIST             all authorizing: ['NAMESPACE_LIST', 'CATALOG_MANAGE_METADATA', 'CATALOG_MANAGE_CONTENT']
  Namespace.DROP   minimal: NAMESPACE_DROP             all authorizing: ['NAMESPACE_DROP', 'CATALOG_MANAGE_CONTENT']
  Table.CREATE     minimal: TABLE_CREATE               all authorizing: ['TABLE_CREATE', 'CATALOG_MANAGE_CONTENT']
  Table.LIST       minimal: TABLE_LIST                 all authorizing: ['TABLE_LIST', 'CATALOG_MANAGE_METADATA', 'CATALOG_MANAGE_CONTENT']
  Table.READ       minimal: TABLE_READ_DATA            all authorizing: ['TABLE_READ_DATA', 'TABLE_READ_PROPERTIES', 'CATALOG_MANAGE_METADATA', 'CATALOG_MANAGE_CONTENT']
  Table.COMMIT     minimal: TABLE_WRITE_DATA           all authorizing: ['TABLE_WRITE_DATA', 'TABLE_WRITE_PROPERTIES', 'CATALOG_MA

### Optional — Necessity pass

Sufficiency (above) proves a privilege is *enough*. Necessity proves it is *required*:
grant the coarse master **minus** the candidate and confirm the action still works, or
grant everything-but-the-candidate and confirm it's blocked. Run only for the
fine-grained privileges that passed sufficiency.

In [8]:
# ── necessity: grant the coarse master, then test WITHOUT the candidate ───────
# For each action whose fine-grained privilege AUTHORIZED, verify that privilege
# is NECESSARY by granting an unrelated single privilege and expecting BLOCKED_PRIV.
print("Necessity check (unrelated-privilege → expect BLOCKED_PRIV):\n")
UNRELATED = "VIEW_LIST"   # a privilege that should not authorize most mutations
for s in SPECS:
    label=f"{s['target']}.{s['action']}"
    fine_auth=[p for p in s["candidates"] if MATRIX.get((s["key"],label,p))=="AUTHORIZED"]
    if not fine_auth: continue
    probe = UNRELATED if UNRELATED not in s["candidates"] else "TABLE_LIST"
    out = probe_privilege(s, probe)
    verdict = "✅ necessary" if out in ("BLOCKED_PRIV","GRANT_INVALID") else f"⚠️ {out} (unexpectedly allowed)"
    print(f"  {label:16} with {probe:12} → {out:14} {verdict}")

Necessity check (unrelated-privilege → expect BLOCKED_PRIV):

  Namespace.CREATE with VIEW_LIST    → ERROR          ⚠️ ERROR (unexpectedly allowed)
  Namespace.LIST   with VIEW_LIST    → BLOCKED_PRIV   ✅ necessary
  Namespace.DROP   with VIEW_LIST    → BLOCKED_PRIV   ✅ necessary
  Table.CREATE     with VIEW_LIST    → BLOCKED_PRIV   ✅ necessary
  Table.LIST       with VIEW_LIST    → BLOCKED_PRIV   ✅ necessary
  Table.READ       with VIEW_LIST    → BLOCKED_PRIV   ✅ necessary
  Table.COMMIT     with VIEW_LIST    → BLOCKED_PRIV   ✅ necessary
  Table.DROP       with VIEW_LIST    → BLOCKED_PRIV   ✅ necessary
  View.CREATE      with VIEW_LIST    → ERROR          ⚠️ ERROR (unexpectedly allowed)
  View.GET         with TABLE_LIST   → ERROR          ⚠️ ERROR (unexpectedly allowed)
  View.DROP        with VIEW_LIST    → BLOCKED_PRIV   ✅ necessary
